# Poverty Map (Population)

## Setup

### Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
!ln -s /content/drive/MyDrive/Colab\ Notebooks/SES-Inference /mydrive

Mounted at /content/drive


### Dependencies

In [ ]:
!pip install geopandas
!pip install rtree
!pip install pyshp

In [6]:
import os
import sys
from shapely.geometry import Point

sys.path.append('/mydrive/libs/')
%set_env PYTHONPATH=/env/python:/mydrive/libs/

%load_ext autoreload
%autoreload 2

from maps import geo
from utils import ios
from facebook.population import FacebookPopulation

env: PYTHONPATH=/env/python:/mydrive/libs/
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Data

### Population data (Facebook)

In [7]:
path = "/mydrive/data/Uganda/"
fn_pop = os.path.join(path, 'population/Facebook/','population_uga_2019-07-01.csv.zip')

In [8]:
gdf_pop = geo.load_as_GeoDataFrame(fn=fn_pop, index_col=None, lat='Lat', lon='Lon', crs=geo.PROJ_DEG)
gdf_pop.head()

,Lat,Lon,Population,geometry
0,3.045383,30.893383,10.745385,POINT (30.89338 3.04538)
1,1.874723,33.588810,6.716464,POINT (33.58881 1.87472)
2,3.068696,32.812557,7.369430,POINT (32.81256 3.06870)
3,3.396389,33.503323,9.413139,POINT (33.50332 3.39639)
4,3.034412,30.961361,9.854125,POINT (30.96136 3.03441)


### Survey Data (clusters)

In [9]:
fn_dhs = '/mydrive/data/Uganda/results/DHS2016_MIS2018_iwi_cluster.csv'

In [10]:
df_dhs = ios.load_csv(fn_dhs, index_col=0)
gdf_dhs = geo.get_GeoDataFrame(df=df_dhs, lat='LATNUM', lon='LONGNUM', crs=geo.PROJ_DEG)
gdf_dhs.head()

,DHSCC,DHSYEAR,DHSCLUST,URBAN_RURA,LATNUM,LONGNUM,SOURCE,ALT_GPS,ALT_DEM,DATUM,mean_iwi,SES,geometry
0,UG,2016,1,1,0.320188,32.568206,GPS,9999.0,1197.0,WGS84,43.300000,rich,POINT (32.56821 0.32019)
1,UG,2016,2,1,0.340653,32.593627,GPS,9999.0,1179.0,WGS84,24.967857,rich,POINT (32.59363 0.34065)
2,UG,2016,3,1,0.313103,32.566556,GPS,9999.0,1189.0,WGS84,34.732000,rich,POINT (32.56656 0.31310)
3,UG,2016,4,1,0.353368,32.558144,GPS,9999.0,1181.0,WGS84,36.060714,rich,POINT (32.55814 0.35337)
4,UG,2016,5,1,0.367388,32.594357,GPS,9999.0,1226.0,WGS84,44.857692,rich,POINT (32.59436 0.36739)


### Very-slow implementation

In [11]:
### Projection to meters
gdf_dhs_proj = geo.get_projection(gdf_dhs, geo.PROJ_MET)
gdf_pop_proj = geo.get_projection(gdf_pop, geo.PROJ_MET)
places_proj = geo.get_STRtree(gdf_pop_proj)

In [12]:
# number of people per cluster (within 30x30 m2 tile)
# number of people within width x width m2

counter = 5
METERS = [geo.MILE_TO_M, geo.KM_TO_M*5, geo.KM_TO_M*10]
for index, row in gdf_dhs_proj.iterrows():

  # num. of people within nearest tile 
  nearest, distance_nearest = geo.find_nearest_place(places_proj, row.geometry)
  population =  gdf_pop_proj[~gdf_pop_proj.intersection(nearest).is_empty].Population.values[0]

  # sum of num of people within meter-square area
  population_area = {}
  for meters in METERS:
    k = round(meters/1000.,1)
    places_in_area = geo.find_places_sqm(gdf_proj=gdf_pop_proj, point_proj=row.geometry, width=meters)
    population_area[k] = places_in_area.Population.sum()

  # update
  df_dhs.loc[index,'population_closest_tile'] = population
  df_dhs.loc[index,'distance_closest_tile'] = distance_nearest
  for k,v in population_area.items():
    df_dhs.loc[index,'population_in_{}km'.format(k)] = v

  print(gdf_dhs.loc[index,'geometry'], population, distance_nearest, geo.convert(nearest, geo.PROJ_MET, geo.PROJ_DEG))
  counter -= 1
  if counter <= 0:
    break

POINT (32.5682063677 0.320187631511) 14.780448360064854 43.570147398668205 POINT (32.56845474243163 0.3204901194838854)
POINT (32.5936271164 0.340653350002) 14.780448360064854 27.958415821090185 POINT (32.5938606262207 0.3407458246250101)
POINT (32.5665559657 0.313103244185) 14.780448360064854 25.76410118678743 POINT (32.5663948059082 0.3129371342972291)
POINT (32.5581439634 0.353368071333) 33.08797166107883 29.287408353467633 POINT (32.55815505981445 0.3531052171723417)
POINT (32.5943566295 0.3673876862129999) 33.08797166107883 26.120939506780157 POINT (32.59454727172852 0.3675244876705222)


In [13]:
df_dhs.drop(columns="geometry").head(5)

,DHSCC,DHSYEAR,DHSCLUST,URBAN_RURA,LATNUM,LONGNUM,SOURCE,ALT_GPS,ALT_DEM,DATUM,mean_iwi,SES,population_closest_tile,distance_closest_tile,population_in_1.6km,population_in_5.0km,population_in_10.0km
0,UG,2016,1,1,0.320188,32.568206,GPS,9999.0,1197.0,WGS84,43.300000,rich,14.780448,43.570147,22366.634497,266328.392079,1.180737e+06
1,UG,2016,2,1,0.340653,32.593627,GPS,9999.0,1179.0,WGS84,24.967857,rich,14.780448,27.958416,19577.204413,238955.658769,1.025650e+06
2,UG,2016,3,1,0.313103,32.566556,GPS,9999.0,1189.0,WGS84,34.732000,rich,14.780448,25.764101,24388.507572,261012.279742,1.153742e+06
3,UG,2016,4,1,0.353368,32.558144,GPS,9999.0,1181.0,WGS84,36.060714,rich,33.087972,29.287408,47960.786056,392824.810726,1.124804e+06
4,UG,2016,5,1,0.367388,32.594357,GPS,9999.0,1226.0,WGS84,44.857692,rich,33.087972,26.120940,27290.527148,255762.523401,9.102187e+05


### Fast implementation (all-in-one)

In [14]:
fb = FacebookPopulation(fn_pop=fn_pop, fn_dhs=fn_dhs)
fb.load_data()
fb.project_data()
fb.delete_nonprojected_variables()

In [15]:
df_dhs_new = fb.update_population_features()
df_dhs_new.head()

100%|██████████| 4/4 [01:28<00:00, 22.03s/it]


,mean_iwi,SES,distance_closest_tile,population_closest_tile,population_in_1.6km,population_in_2.0km,population_in_5.0km,population_in_10.0km
0,43.300000,rich,43.570147,14.780448,22366.634497,37133.430237,266328.392079,1.180737e+06
1,24.967857,rich,27.958416,14.780448,19577.204413,33030.175271,238955.658769,1.025650e+06
2,34.732000,rich,25.764101,14.780448,24388.507572,37241.554805,261012.279742,1.153742e+06
3,36.060714,rich,29.287408,33.087972,47960.786056,71201.123993,392824.810726,1.124804e+06
4,44.857692,rich,26.120940,33.087972,27290.527148,40174.711688,255762.523401,9.102187e+05
